# Data Collection - UK Energy Demand Forecaster 

This notebook collects all raw data required for the project from 3 sources:

1. **Electricity demand data**: Half-hourly national demand from NESO (National Energy System Operator), covering 2023–2025
2. **Weather data**: Hourly weather observations for London from the Open-Meteo API
3. **Bank holidays**:England & Wales bank holidays from the GOV.UK API
4. **School holidays**:Created manually as a separate CSV (see data/raw/school_holidays.csv)

**Note:** These downloads were run in June 2026. The NESO data portal URLs may change over time. If the download cells fail, the CSVs can be downloaded manually from https://www.neso.energy/data-portal/historic-demand-data

In [1]:
import requests
import os
import pandas as pd
import json
from pathlib import Path   


CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw data folder: {RAW_DIR}")


Raw data folder: c:\Users\marco\Documents\PROJECTS\UK-Energy-forecaster\data\raw


## 1. Electricity Demand Data (NESO)

Downloading three full calendar years (2023–2025) of half-hourly electricity demand data from the NESO data portal. Each year contains approx. 17,520 rows (365 days × 48 settlement periods per day).

Quick checks: row counts, date ranges, TSD (Transmission System Demand) value ranges and missing values.

**Observation:** The date formats differ across years, 2023 uses `01-Jan-23`, 2024 uses `01-JAN-2024`, and 2025 uses `2025-01-01`. This will need harmonising during the data loading phase.

In [ ]:



#Downloading 2023, 2024, 2025 demand data
base_url = "https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource"

files = {
    "demanddata_2023.csv": "bf5ab335-9b40-4ea4-b93a-ab4af7bce003",
    "demanddata_2024.csv": "f6d02c0f-957b-48cb-82ee-09003f2ba759",
    "demanddata_2025.csv": "b2bde559-3455-4021-b179-dfe60c0337b0",
}

for filename, resource_id in files.items():
    url = f"{base_url}/{resource_id}/download/{filename}"
    print(f"Downloading {filename}...")
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    with open(RAW_DIR / filename, "wb") as f:
        f.write(response.content)
    print(f"  Saved! ({len(response.content) / 1024 / 1024:.1f} MB)")

print("Done!")

df = pd.read_csv(RAW_DIR / "demanddata_2023.csv")
print(df.shape)
print(df.columns.tolist())
df.head()



for year in [2023, 2024, 2025]:
    df = pd.read_csv(RAW_DIR / f"demanddata_{year}.csv")
    print(f"\n{year}: {df.shape[0]} rows, {df.shape[1]} columns")
    print(f"  Date range: {df['SETTLEMENT_DATE'].iloc[0]} to {df['SETTLEMENT_DATE'].iloc[-1]}")
    print(f"  TSD range: {df['TSD'].min():.0f} MW to {df['TSD'].max():.0f} MW")
    print(f"  Missing TSD values: {df['TSD'].isna().sum()}")

  Saved! (1.7 MB)
  Saved! (1.7 MB)
  Saved! (1.8 MB)
Done!
(17520, 22)
['SETTLEMENT_DATE', 'SETTLEMENT_PERIOD', 'ND', 'TSD', 'ENGLAND_WALES_DEMAND', 'EMBEDDED_WIND_GENERATION', 'EMBEDDED_WIND_CAPACITY', 'EMBEDDED_SOLAR_GENERATION', 'EMBEDDED_SOLAR_CAPACITY', 'NON_BM_STOR', 'PUMP_STORAGE_PUMPING', 'SCOTTISH_TRANSFER', 'IFA_FLOW', 'IFA2_FLOW', 'BRITNED_FLOW', 'MOYLE_FLOW', 'EAST_WEST_FLOW', 'NEMO_FLOW', 'NSL_FLOW', 'ELECLINK_FLOW', 'VIKING_FLOW', 'GREENLINK_FLOW']

2023: 17520 rows, 22 columns
  Date range: 01-Jan-23 to 31-Dec-23
  TSD range: 16513 MW to 46863 MW
  Missing TSD values: 0

2024: 17568 rows, 22 columns
  Date range: 01-JAN-2024 to 31-DEC-2024
  TSD range: 17138 MW to 47507 MW
  Missing TSD values: 0

2025: 17520 rows, 22 columns
  Date range: 01-JAN-2025 to 31-DEC-2025
  TSD range: 15297 MW to 47760 MW
  Missing TSD values: 0


## 2. Weather Data (Open-Meteo API)

Pulling hourly weather observations for London (51.51°N, 0.13°W) for the same 2023–2025 period. London is used as the representative location since the South East accounts for the largest share of UK electricity demand.

Variables collected:
- **temperature_2m**: Air temperature at 2m height (°C)
- **wind_speed_10m**: Wind speed at 10m height (km/h)
- **relative_humidity_2m**: Relative humidity (%)
- **precipitation**: Precipitation (mm)
- **cloud_cover**: Cloud cover (%)
- **direct_radiation**: Direct solar radiation (W/m²)

In [ ]:

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 51.51,
    "longitude": -0.13,
    "start_date": "2023-01-01",
    "end_date": "2025-12-31",
    "hourly": "temperature_2m,wind_speed_10m,relative_humidity_2m,precipitation,cloud_cover,direct_radiation",
    "timezone": "Europe/London"
}

response = requests.get(url, params=params, timeout=120)
response.raise_for_status()
data = response.json()

weather = pd.DataFrame(data["hourly"])
weather.to_csv(RAW_DIR / "weather_london_2023_2025.csv", index=False)
print(f"Weather data: {weather.shape}")
weather.head()
print(weather.columns.tolist())

Weather data: (26304, 7)
['time', 'temperature_2m', 'wind_speed_10m', 'relative_humidity_2m', 'precipitation', 'cloud_cover', 'direct_radiation']


## 3. Bank Holidays (GOV.UK API)

Downloading the official England & Wales bank holiday dates from the GOV.UK JSON API. These will be used as a binary feature in the model. Electricity demand typically drops 15–20% on bank holidays due to reduced commercial and industrial activity.

In [ ]:
url = "https://www.gov.uk/bank-holidays.json"
response = requests.get(url, timeout=120)
response.raise_for_status()
bank_holidays = response.json()

# England and Wales
england_holidays = bank_holidays["england-and-wales"]["events"]
bh_df = pd.DataFrame(england_holidays)
bh_df.to_csv(RAW_DIR / "bank_holidays.csv", index=False)
print(f"Bank holidays: {bh_df.shape}")
bh_df.head()
print(bh_df.columns.tolist())

Bank holidays: (83, 4)
['title', 'date', 'notes', 'bunting']


## 4. School Holidays

School holiday dates for England (2023–2025) have been compiled manually from GOV.UK term date publications and saved as `data/raw/school_holidays.csv`. Approximate dates are used as the effect on energy demand is gradual rather than sharp.

## Summary

All raw data has been collected and saved to `data/raw/`:


| demanddata_2023.csv | NESO | 17,520 | Jan–Dec 2023 |
| demanddata_2024.csv | NESO | 17,568 | Jan–Dec 2024 |
| demanddata_2025.csv | NESO | 17,520 | Jan–Dec 2025 |
| weather_london_2023_2025.csv | Open-Meteo | 26,304 | Jan 2023–Dec 2025 |
| bank_holidays.csv | GOV.UK | 83 | Various |
| school_holidays.csv | Manual | 18 | 2023–2025 |

**Next step:** Loading all data into a SQLite database (see notebook 02).